In [ ]:
import os                                  
os.environ['KAGGLE_API_TOKEN'] = "KGAT_50878a8fd2ebaaceae41390d55e16727"
                                                                            
!pip install -q kaggle --upgrade                                
!kaggle datasets download -d iamsouravbanerjee/indian-food-images-dataset
!unzip -q indian-food-images-dataset.zip -d master_data

print("Master dataset downloaded and extracted!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.4/95.4 kB 1.9 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/iamsouravbanerjee/indian-food-images-dataset
License(s): other
100% 355M/355M [00:02<00:00, 149MB/s]

Master dataset downloaded and extracted!


In [2]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

mithai_classes = [
    'basundi', 'cham_cham', 'gajar_ka_halwa', 'ghevar', 'gulab_jamun',
    'imarti', 'jalebi', 'kalakand', 'malapua', 'mysore_pak',
    'ras_malai', 'rasgulla', 'sandesh', 'shrikhand', 'sohan_papdi'
]

base_dir = Path("sweets_dataset")
train_dir = base_dir / "train"
val_dir = base_dir / "val"                                                    
master_dir = Path("master_data")
source_dir = None                                                       
for path in master_dir.rglob('basundi'):
    if path.is_dir():
        source_dir = path.parent
        break

if source_dir is None:
    print("Error: Could not find the image folders! The dataset might not have downloaded correctly.")
else:
    print(f"Found the image directories at: {source_dir}\n")

                                                
    for dir_path in [train_dir, val_dir]:
        for cls in mithai_classes:
            os.makedirs(dir_path / cls, exist_ok=True)

                                     
    total_copied = 0
    for cls in mithai_classes:
        cls_path = source_dir / cls
        if not cls_path.exists():
            print(f"Missing class folder: {cls}")
            continue

                                                                  
        images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG']:
            images.extend(list(cls_path.glob(ext)))

        if len(images) == 0:
             print(f"Warning: No images found in {cls_path}")
             continue

                                                 
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

        for img in train_imgs:
            shutil.copy(img, train_dir / cls / img.name)
            total_copied += 1
        for img in val_imgs:
            shutil.copy(img, val_dir / cls / img.name)
            total_copied += 1

        print(f"Copied {len(images)} images for {cls}")

    print(f"\nPrepared {total_copied} total images. You are ready to train.")

Found the image directories at: master_data/Indian Food Images/Indian Food Images

Copied 50 images for basundi
Copied 50 images for cham_cham
Copied 50 images for gajar_ka_halwa
Copied 50 images for ghevar
Copied 50 images for gulab_jamun
Copied 50 images for imarti
Copied 50 images for jalebi
Copied 50 images for kalakand
Copied 50 images for malapua
Copied 50 images for mysore_pak
Copied 50 images for ras_malai
Copied 50 images for rasgulla
Copied 50 images for sandesh
Copied 50 images for shrikhand
Copied 50 images for sohan_papdi

Prepared 750 total images. You are ready to train.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from google.colab import files                                     

                              
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

              
train_data = datasets.ImageFolder('sweets_dataset/train', transform=train_transforms)
val_data = datasets.ImageFolder('sweets_dataset/val', transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

                     
model = models.mobilenet_v3_small(weights='DEFAULT')
for param in model.parameters():
    param.requires_grad = False                      

num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, len(train_data.classes))
model = model.to(device)

                       
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

                  
epochs = 10
best_acc = 0.0

print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

                          
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'mithai_model.pth')

print("Training complete! Best model saved as 'mithai_model.pth'")

                                                 
files.download('mithai_model.pth')
print("Download started! You can now use this file locally.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 85.2MB/s]


Starting training...
Epoch 1/10 - Loss: 2.4508 - Val Acc: 38.67%
Epoch 2/10 - Loss: 1.8328 - Val Acc: 54.00%
Epoch 3/10 - Loss: 1.5131 - Val Acc: 58.00%
Epoch 4/10 - Loss: 1.3171 - Val Acc: 60.67%
Epoch 5/10 - Loss: 1.1812 - Val Acc: 62.67%
Epoch 6/10 - Loss: 1.0698 - Val Acc: 65.33%
Epoch 7/10 - Loss: 1.0417 - Val Acc: 66.00%
Epoch 8/10 - Loss: 0.9483 - Val Acc: 66.67%
Epoch 9/10 - Loss: 0.8721 - Val Acc: 68.00%
Epoch 10/10 - Loss: 0.8822 - Val Acc: 68.67%
Training complete! Best model saved as 'mithai_model.pth'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! You can now use this file locally.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from google.colab import files

                              
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

              
train_data = datasets.ImageFolder('sweets_dataset/train', transform=train_transforms)
val_data = datasets.ImageFolder('sweets_dataset/val', transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

                     
model = models.mobilenet_v3_small(weights='DEFAULT')
for param in model.parameters():
    param.requires_grad = False                                

num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, len(train_data.classes))
model = model.to(device)

                                                           
                                                   
                                                           
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

epochs_phase1 = 10
best_acc = 0.0

print("--- Starting Phase 1: Classifier Warm-up ---")
for epoch in range(epochs_phase1):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

                    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Phase 1 - Epoch {epoch+1}/{epochs_phase1} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'mithai_model_best.pth')

                                                           
                                                          
                                                           
print("\n--- Starting Phase 2: Fine-Tuning ---")

                                                     
for param in model.features[-4:].parameters():
    param.requires_grad = True

                                                                           
                                                                
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

epochs_phase2 = 15

for epoch in range(epochs_phase2):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_ft.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item()

                    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Phase 2 - Epoch {epoch+1}/{epochs_phase2} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
                                                                     
        torch.save(model.state_dict(), 'mithai_model_best.pth')

print(f"\nTraining complete! Absolute best validation accuracy achieved: {best_acc:.2f}%")

                                                    
files.download('mithai_model_best.pth')
print("Download started! You can now use 'mithai_model_best.pth' locally for your Streamlit app.")

--- Starting Phase 1: Classifier Warm-up ---
Phase 1 - Epoch 1/10 - Loss: 2.5038 - Val Acc: 35.33%
Phase 1 - Epoch 2/10 - Loss: 1.9114 - Val Acc: 50.00%
Phase 1 - Epoch 3/10 - Loss: 1.5294 - Val Acc: 56.00%
Phase 1 - Epoch 4/10 - Loss: 1.3336 - Val Acc: 61.33%
Phase 1 - Epoch 5/10 - Loss: 1.1820 - Val Acc: 57.33%
Phase 1 - Epoch 6/10 - Loss: 1.0615 - Val Acc: 60.67%
Phase 1 - Epoch 7/10 - Loss: 0.9922 - Val Acc: 65.33%
Phase 1 - Epoch 8/10 - Loss: 0.9702 - Val Acc: 66.00%
Phase 1 - Epoch 9/10 - Loss: 0.9122 - Val Acc: 67.33%
Phase 1 - Epoch 10/10 - Loss: 0.8921 - Val Acc: 69.33%

--- Starting Phase 2: Fine-Tuning ---
Phase 2 - Epoch 1/15 - Loss: 0.7729 - Val Acc: 72.00%
Phase 2 - Epoch 2/15 - Loss: 0.7354 - Val Acc: 74.00%
Phase 2 - Epoch 3/15 - Loss: 0.6776 - Val Acc: 75.33%
Phase 2 - Epoch 4/15 - Loss: 0.6260 - Val Acc: 74.67%
Phase 2 - Epoch 5/15 - Loss: 0.5653 - Val Acc: 74.00%
Phase 2 - Epoch 6/15 - Loss: 0.5885 - Val Acc: 73.33%
Phase 2 - Epoch 7/15 - Loss: 0.5639 - Val Acc: 74.0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! You can now use 'mithai_model_best.pth' locally for your Streamlit app.
